## Aproximación por serie de Taylor de $f(x) = x^{-3}(\sin x - x)$

### Derivación

Partiendo de la serie de Taylor del seno:
$\sin x = \sum_{i=0}^{\infty} \frac{(-1)^i\, x^{2i+1}}{(2i+1)!}$

Se tiene que:
$\sin x - x = \sum_{i=1}^{\infty} \frac{(-1)^i\, x^{2i+1}}{(2i+1)!}$

Dividiendo por $x^3$ (con $x \neq 0$) y reindexando con $j = i - 1$:

$f(x) = x^{-3}(\sin x - x) = \sum_{j=0}^{\infty} \frac{(-1)^{j+1}\, x^{2j}}{(2j+3)!} = -\frac{1}{3!} + \frac{x^2}{5!} - \frac{x^4}{7!} + \cdots$

Además podemos evitar el caso en que $x = 0$ eliminando analíticamente ese término del denominador. De esta forma la serie converge para todo $x$ y es bien comportada cerca del origen.

### Recurrencia entre términos consecutivos

Siendo $t_j = \dfrac{(-1)^{j+1}\, x^{2j}}{(2j+3)!}$:

$t_{j+1} = t_j \cdot \frac{-x^2}{(2j+5)(2j+4)}$

Con `denom` $= 2j + 5$ (arranca en 5, incrementa de 2 en 2):

$t_{\text{new}} = t_{\text{prev}} \cdot \frac{-x^2}{\texttt{denom} \cdot (\texttt{denom} - 1)}$

Esto evita calcular potencias y factoriales desde cero en cada paso.

### Implementación — con precisión configurable

La función `f_aprox(x, tol, precision)` acepta un string de precisión:
- `"Float16"` → 16 bits (~3 dígitos decimales, épsilon de máquina ≈ 4.88e-4)
- `"Float32"` → 32 bits (~7 dígitos decimales, épsilon de máquina ≈ 1.19e-7)
- `"Float64"` → 64 bits (~15 dígitos decimales, épsilon de máquina ≈ 2.22e-16)

**Todos** los cálculos internos (constantes, acumuladores, operaciones) se realizan
en el tipo indicado, de modo que los errores de redondeo y representación son
exactamente los propios de esa aritmética.

In [15]:
"""
    f_aprox(a, tol, precision="Float64")

Aproxima f(x) = x⁻³·(sin(x) - x) mediante la serie alternada de Taylor:

    f(x) = Σ_{j=0}^{∞}  (-1)^{j+1} · x^{2j} / (2j+3)!
         = -1/3! + x²/5! - x⁴/7! + ...

usando recurrencia entre términos consecutivos.

# Argumentos
- `a`         : ángulo en grados (degrees); se convierte a radianes internamente
- `tol`       : tolerancia de parada — |término| < tol
- `precision` : string con el tipo flotante a usar: "Float16", "Float32" o "Float64"

Todos los cálculos internos se realizan en el tipo especificado, por lo que
los errores de redondeo y representación son los propios de esa aritmética.
"""
function f_aprox(a, tol, precision="Float64", max_iterations = 1000)

    # Selección del tipo numérico según el string de precisión
    T = if precision == "Float32"
            Float32
        elseif precision == "Float64"
            Float64
        else
            error("Precisión no reconocida: '$precision'. Usa \"Float16\", \"Float32\" o \"Float64\".")
        end

    # Conversión de grados a radianes en precisión T
    x = T(a) / (T(180) / T(pi))

    # Convertir TODAS las constantes y la entrada al tipo T
    x_cuad    = x ^ T(2)          # x² en precisión T
    term  = T(-1) / T(6)         # t₀ = -1/3! = -1/6  en precisión T
    res   = term
    denom = T(5)                  # primer factor del denominador recurrente
    tol_T = T(tol)

    
    converged = false
    count_iterations = 0

    for _ in 1:max_iterations
        # tⱼ₊₁ = tⱼ · (−x²) / [(denom)·(denom−1)]  — todo en tipo T
        term = (-(term * x_cuad)) / (denom * (denom - T(1)))
        res  = res + term
        count_iterations += 1

        if abs(term) < tol_T
            converged = true
            break
        end
        
        denom = denom + T(2)
    end

    if !converged
        println("[WARNING]: Terminó por máximo de iteraciones ($max_iterations)")
    else
        println("[INFO]: Convergió en $count_iterations iteraciones")
    end

    return res, count_iterations
end


f_aprox

### Tabla comparativa de resultados por precisión

Se usa [`PrettyTables.jl`](https://github.com/ronisbr/PrettyTables.jl) para mostrar en una tabla bien formateada los valores de referencia y los errores absolutos de cada precisión para los ángulos de interés.

In [16]:
# Valor de referencia en BigFloat — recibe grados, convierte a radianes
function f_ref(a)
    xb = BigFloat(a) / (BigFloat(180) / BigFloat(pi))
    (sin(xb) - xb) / xb^3
end

f_ref (generic function with 1 method)

In [17]:
import Pkg; 
Pkg.add("PrettyTables")
Pkg.status("PrettyTables")


   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


Status `~/.julia/environments/v1.12/Project.toml`
  [08abe8d2] PrettyTables v3.3.2


In [19]:
using Printf
using PrettyTables

angulos = [30, 390, 750, 1110, 1470, 1830, 2190, 2550, 2910]
tol     = 1e-6

# Calcular valores e iteraciones necesitadas
f32_val_iter = []
f64_val_iter = []

refs   = [Float64(f_ref(a)) for a in angulos]

for a in angulos    
    appr, iters = f_aprox(a, tol, "Float32")
    push!(f32_val_iter, (Float64(appr),iters))
    
    appr, iters = f_aprox(a, tol, "Float64")
    push!(f64_val_iter, (Float64(appr),iters))
end

# Obtenemos errores absolutos de cada aproximación
err32 = [abs(x[1] - r) for (x, r) in zip(f32_val_iter, refs)]
err64 = [abs(x[1] - r) for (x, r) in zip(f64_val_iter, refs)]

# Armar la tabla
data = hcat(angulos, refs, first.(f32_val_iter), first.(f64_val_iter), err32, err64)

pretty_table(
    data,
    backend       = :html,
    column_labels = [
        "Ángulo (°)", "Referencia (BigFloat)", "Float32", "Float64", "Err abs. F32", "Err abs. F64"
    ],
    formatters = [(v, i, j) -> j in 2:8 ? @sprintf("%.6e", v) : v],
    alignment  = [:r, :r, :r, :r, :r, :r],
)


[INFO]: Convergió en 3 iteraciones
[INFO]: Convergió en 3 iteraciones
[INFO]: Convergió en 11 iteraciones
[INFO]: Convergió en 11 iteraciones
[INFO]: Convergió en 18 iteraciones
[INFO]: Convergió en 18 iteraciones
[INFO]: Convergió en 26 iteraciones
[INFO]: Convergió en 26 iteraciones
[INFO]: Convergió en 34 iteraciones
[INFO]: Convergió en 34 iteraciones
[INFO]: Convergió en 43 iteraciones
[INFO]: Convergió en 43 iteraciones
[INFO]: Convergió en 51 iteraciones
[INFO]: Convergió en 51 iteraciones
[INFO]: Convergió en 59 iteraciones
[INFO]: Convergió en 59 iteraciones
[INFO]: Convergió en 67 iteraciones
[INFO]: Convergió en 67 iteraciones


Ángulo (°),Referencia (BigFloat),Float32,Float64,Err abs. F32,Err abs. F64
30.0,-1.643969e-01,-1.643969e-01,-1.643969e-01,4.619736e-09,1.412766e-10
390.0,-1.999779e-02,-1.999781e-02,-1.999778e-02,1.558293e-08,8.498868e-09
750.0,-5.613178e-03,-5.613152e-03,-5.613253e-03,2.516841e-08,7.578447e-08
1110.0,-2.595634e-03,-2.574422e-03,-2.595706e-03,2.121187e-05,7.171252e-08
1470.0,-1.489578e-03,-5.663990e-02,-1.489666e-03,5.515032e-02,8.796902e-08
1830.0,-9.649184e-04,-4.403520e+00,-9.649078e-04,4.402555e+00,1.064143e-08
2190.0,-6.755204e-04,-3.673624e+03,-6.776293e-04,3.673624e+03,2.108904e-06
2550.0,-4.991812e-04,-3.800284e+05,-2.124305e-04,3.800284e+05,2.867507e-04
2910.0,-3.838510e-04,-1.269679e+08,2.192821e-01,1.269679e+08,2.196660e-01


> [!WARNING]
> Hasta aquí vienen funcionando las celdas, luego no probé

### Gráfico para contrastar errores absolutos
Cada grupo de barras corresponde a un valor de entrada; dentro del grupo se comparan los errores absolutos de las tres precisiones.

In [ ]:
import Pkg; Pkg.add("Plots")
Pkg.status("Plots")


   Resolving package versions...
   Installed Libmount_jll ───────── v2.41.3+0
   Installed Opus_jll ───────────── v1.6.1+0
   Installed ConcurrentUtilities ── v2.5.1
   Installed GR ─────────────────── v0.73.24
   Installed GR_jll ─────────────── v0.73.24+0
   Installed libva_jll ──────────── v2.23.0+0
   Installed Cairo_jll ──────────── v1.18.6+0
   Installed HTTP ───────────────── v1.11.0
   Installed MbedTLS ────────────── v1.1.10
   Installed XZ_jll ─────────────── v5.8.3+0
   Installed Xorg_libxkbfile_jll ── v1.2.0+0
   Installed Xorg_libXinerama_jll ─ v1.1.7+0
   Installed PtrArrays ──────────── v1.4.0
   Installed FreeType2_jll ──────── v2.14.3+1
   Installed DataStructures ─────── v0.19.4
   Installed Qt6Svg_jll ─────────── v6.10.2+0
   Installed libpng_jll ─────────── v1.6.56+0
   Installed Libuuid_jll ────────── v2.41.3+0
   Installed Qt6Declarative_jll ─── v6.10.2+1
   Installed Xorg_libXext_jll ───── v1.3.8+0
   Installed Qt6ShaderTools_jll ─── v6.10.2+1
   Installed Plots

Status `~/.julia/environments/v1.12/Project.toml`
  [91a5bcdd] Plots v1.41.6


In [ ]:

using Plots
gr()

angulos = [30, 390, 750, 1110, 1470, 1830, 2190, 2550, 2910]
tol     = 1e-6

etiquetas = string.(angulos) .* "°"
n = length(angulos)
x = 1:n
ancho = 0.25

p1 = bar(
    x .- ancho,  err32;
    bar_width   = ancho,
    label       = "Float32",
    color       = :tomato,
    alpha       = 0.85,
    xlabel      = "Ángulo de entrada",
    ylabel      = "Error absoluto",
    title       = "Errores absolutos por precisión",
    xticks      = (x, etiquetas),
    yscale      = :log10,
    legend      = :topright,
    size        = (900, 480),
    grid        = true,
    gridcolor   = :lightgray,
    framestyle  = :box,
)
bar!(p1, x .+ ancho, err64; bar_width=ancho, label="Float64", color=:mediumseagreen, alpha=0.85)

display(p1)


LoadError: UndefVarError: `err32` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### Gráfico para visualizar errores relativos por ángulo
El error relativo normaliza el error absoluto respecto al valor de referencia, permitiendo comparar la precisión independientemente de la magnitud de `f(x)`.

### Cancelación catastrófica: expresión directa vs. serie de Taylor

Para valores muy pequeños de $x$, la expresión directa `(sin(x) - x) / x^3` sufre
**cancelación catastrófica** porque:

- `sin(x) ≈ x` cuando $x \to 0$, por lo que `sin(x) - x ≈ 0`
- La resta de dos números muy similares destruye los dígitos significativos
- Dividir por `x^3` (muy pequeño) **amplifica** el error residual

Este efecto es **mucho más severo** con Float16 y Float32 porque tienen menos bits
de mantisa para absorber la pérdida de significancia.

La aproximación por serie evita este problema porque la cancelación ya fue
realizada **simbólicamente** al derivar la serie.

In [ ]:
println("Cancelación catastrófica en la expresión directa por precisión:")
println("─"^95)
println(rpad("a (°)", 10),
        rpad("Precisión", 12),
        rpad("Serie (estable)", 24),
        rpad("Directa (sin(x)-x)/x³", 28),
        "Error directo")
println("─"^95)

ref_limit = -1/6   # límite cuando a→0: f(0) = -1/6

# Ángulos pequeños en grados (equivalen a x radianes muy pequeños)
angulos_cc = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]

for a in angulos_cc
    for p in ["Float16", "Float32", "Float64"]
        T  = if p == "Float16" Float16 elseif p == "Float32" Float32 else Float64 end
        xT = T(a) / (T(180) / T(pi))   # conversión en precisión T
        serie   = Float64(f_aprox(a, 1e-6, p))
        directo = Float64((sin(xT) - xT) / xT^3)
        err_dir = abs(directo - ref_limit)
        println(rpad(string(a)*"°", 10),
                rpad(p, 12),
                rpad(string(serie),   24),
                rpad(string(directo), 28),
                err_dir)
    end
    println("─"^95)
end

println("Valor límite esperado: f(0) = lim_{x→0} (sin(x)-x)/x³ = -1/6 ≈ ", -1/6)


### Épsilon de máquina y límites de representación

El **épsilon de máquina** (`eps(T)`) es el error de representación fundamental de cada tipo.
Ninguna aproximación puede superar esta barrera de precisión, sin importar cuántos
términos de la serie se sumen.

In [ ]:
println("Épsilon de máquina por tipo:")
println("  Float16 → eps = ", eps(Float16), "  (≈ ", Float64(eps(Float16)), ")")
println("  Float32 → eps = ", eps(Float32), "  (≈ ", Float64(eps(Float32)), ")")
println("  Float64 → eps = ", eps(Float64))
println()
println("Bits de mantisa:")
println("  Float16 → ", precision(Float16), " bits significativos")
println("  Float32 → ", precision(Float32), " bits significativos")
println("  Float64 → ", precision(Float64), " bits significativos")
